In [ ]:
%pip -q install "mcp>=1.9,<2" pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.6 MB/s eta 0:00:00


In [ ]:
# ================================================================
# EXPERIMENT 06: AGENTIC DEVELOPER RELEASE-READINESS SYSTEM USING MCP
# Domain: Developer Tools | Google Colab
# ================================================================

import ast
import asyncio
import json
import logging
import re
import statistics
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

# -------------------------
# 1. Configuration
# -------------------------

PROJECT_DIR = Path("/content/release_readiness_project")
LOG_DIR = Path("/content/mcp_logs")
SERVER_FILE = Path("/content/mcp_release_readiness_server.py")
SERVER_STDERR_FILE = LOG_DIR / "mcp_server_stderr.log"
LOG_FILE = LOG_DIR / "execution.log"
TRACE_FILE = LOG_DIR / "mcp_traces.jsonl"
REPORT_FILE = LOG_DIR / "developer_automation_report.json"
COMPLETE_REPORT_FILE = LOG_DIR / "experiment_06_complete_report.json"
SUMMARY_FILE = LOG_DIR / "experiment_06_execution_summary.json"
ARTIFACT_ZIP = Path("/content/experiment_06_artifacts.zip")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

for f in (LOG_FILE, TRACE_FILE, SERVER_STDERR_FILE):
    f.write_text("", encoding="utf-8")

logging.basicConfig(
    filename=str(LOG_FILE),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)
logger = logging.getLogger("MCP-Release-Readiness")
logger.info("Experiment 06 started")

# -------------------------
# 2. Create custom sample project
# -------------------------
# Custom use case: release-readiness inspection of a Python pricing package.
# The unused import, missing docstring and TODO are intentional findings.

pricing_code = '''\
import math  # Deliberate unused-import finding

def calculate_subtotal(unit_price, quantity):
    """Calculate the total before discounts and tax."""
    if unit_price < 0:
        raise ValueError("unit_price cannot be negative")
    if quantity < 0:
        raise ValueError("quantity cannot be negative")
    return round(unit_price * quantity, 2)


def apply_discount(total, discount_percent):
    """Apply a percentage discount to a total."""
    if not 0 <= discount_percent <= 100:
        raise ValueError("discount_percent must be between 0 and 100")
    return round(total * (1 - discount_percent / 100), 2)


def calculate_final_price(unit_price, quantity, discount_percent=0):
    """Calculate the final price after applying a discount."""
    subtotal = calculate_subtotal(unit_price, quantity)
    return apply_discount(subtotal, discount_percent)


def add_tax(total, tax_rate=0.18):
    # Deliberately missing docstring for quality analysis.
    if total < 0 or tax_rate < 0:
        raise ValueError("total and tax_rate must be non-negative")
    return round(total * (1 + tax_rate), 2)

# TODO: add locale-aware currency formatting before production release.
'''

app_code = '''\
from pricing import calculate_final_price, add_tax


def quote(unit_price, quantity, discount_percent=0):
    """Return a final price including tax."""
    discounted = calculate_final_price(
        unit_price, quantity, discount_percent
    )
    return add_tax(discounted)


if __name__ == "__main__":
    print("Sample release-readiness quote:", quote(100, 2, 10))
'''

test_code = '''\
import pytest
from pricing import (
    calculate_subtotal, apply_discount,
    calculate_final_price, add_tax
)
from app import quote


def test_calculate_subtotal():
    assert calculate_subtotal(10, 3) == 30.00


def test_negative_price_rejected():
    with pytest.raises(ValueError):
        calculate_subtotal(-1, 2)


def test_negative_quantity_rejected():
    with pytest.raises(ValueError):
        calculate_subtotal(10, -2)


def test_apply_discount():
    assert apply_discount(100, 15) == 85.00


def test_invalid_discount_rejected():
    with pytest.raises(ValueError):
        apply_discount(100, 120)


def test_final_price():
    assert calculate_final_price(50, 2, 10) == 90.00


def test_add_tax():
    assert add_tax(10, 0.18) == 11.80


def test_app_quote():
    assert quote(100, 1, 10) == 106.20
'''

readme_text = '''\
# Pricing Package — Release Readiness Demo

A small Python package for demonstrating MCP-based developer automation.

Files:
- pricing.py: subtotal, discount, final-price and tax logic.
- app.py: a small entry point for price quotes.
- test_pricing.py: automated unit tests.

Release checks include file inventory, source analysis, TODO search,
README inspection, and test execution. Static analysis deliberately reports
an unused import, a missing docstring, and a TODO comment.
'''

(PROJECT_DIR / "pricing.py").write_text(pricing_code, encoding="utf-8")
(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")
(PROJECT_DIR / "test_pricing.py").write_text(test_code, encoding="utf-8")
(PROJECT_DIR / "README.md").write_text(readme_text, encoding="utf-8")

print("=" * 72)
print("EXPERIMENT 06: AGENTIC DEVELOPER RELEASE-READINESS SYSTEM")
print("=" * 72)
print("Domain: Developer Tools")
print("Use case: Python pricing package release-readiness checks")
print("Project:", PROJECT_DIR)
print("Files:", sorted(p.name for p in PROJECT_DIR.iterdir() if p.is_file()))

# -------------------------
# 3. Create actual MCP server
# -------------------------
# This server runs as a separate process over stdio.
# Keep stdout reserved for MCP protocol messages.

server_code = '''\
import ast
import json
import re
import subprocess
import sys
import time
from pathlib import Path
from mcp.server.fastmcp import FastMCP

PROJECT_DIR = Path("/content/release_readiness_project").resolve()
mcp = FastMCP("Agentic Developer Release Readiness Server")


def safe_project_path(relative_path: str) -> Path:
    requested = (PROJECT_DIR / relative_path).resolve()
    if requested == PROJECT_DIR or (
        PROJECT_DIR not in requested.parents
    ):
        raise PermissionError("Path outside project directory")
    return requested


def as_json(payload: dict) -> str:
    return json.dumps(payload, ensure_ascii=False)


@mcp.tool()
def list_files() -> str:
    """List files in the sample release-readiness project."""
    files = sorted(
        p.name for p in PROJECT_DIR.iterdir() if p.is_file()
    )
    return as_json({
        "status": "success",
        "files": files,
        "count": len(files)
    })


@mcp.tool()
def read_file(file_path: str) -> str:
    """Read a text file inside the sample project."""
    try:
        path = safe_project_path(file_path)
    except PermissionError as exc:
        return as_json({
            "status": "blocked",
            "error_category": "security_error",
            "message": str(exc)
        })

    if not path.exists() or not path.is_file():
        return as_json({
            "status": "error",
            "error_category": "input_error",
            "message": f"File not found: {file_path}"
        })

    try:
        content = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return as_json({
            "status": "error",
            "error_category": "input_error",
            "message": "File is not UTF-8 text"
        })

    return as_json({
        "status": "success",
        "file": file_path,
        "content": content,
        "characters": len(content)
    })


@mcp.tool()
def search_code(keyword: str) -> str:
    """Search for a keyword across Python project files."""
    matches = []
    for path in sorted(PROJECT_DIR.glob("*.py")):
        for line_no, line in enumerate(
            path.read_text(encoding="utf-8").splitlines(), start=1
        ):
            if keyword.casefold() in line.casefold():
                matches.append({
                    "file": path.name,
                    "line": line_no,
                    "content": line.strip()
                })

    return as_json({
        "status": "success",
        "keyword": keyword,
        "match_count": len(matches),
        "matches": matches
    })


@mcp.tool()
def analyze_python(file_path: str) -> str:
    """Run AST-based syntax and code-quality analysis."""
    try:
        path = safe_project_path(file_path)
    except PermissionError as exc:
        return as_json({
            "status": "blocked",
            "error_category": "security_error",
            "message": str(exc)
        })

    if not path.exists() or not path.is_file():
        return as_json({
            "status": "error",
            "error_category": "input_error",
            "message": f"File not found: {file_path}"
        })

    if path.suffix != ".py":
        return as_json({
            "status": "error",
            "error_category": "input_error",
            "message": "Only .py files can be analyzed"
        })

    source = path.read_text(encoding="utf-8")

    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return as_json({
            "status": "error",
            "error_category": "input_error",
            "syntax": "FAIL",
            "message": f"Syntax error at line {exc.lineno}: {exc.msg}"
        })

    functions, classes, imports, issues = [], [], [], []
    referenced_names = {
        n.id for n in ast.walk(tree) if isinstance(n, ast.Name)
    }

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append(node.name)
            if ast.get_docstring(node) is None:
                issues.append(
                    f"Function '{node.name}' has no docstring"
                )

        elif isinstance(node, ast.ClassDef):
            classes.append(node.name)

        elif isinstance(node, ast.Import):
            for alias in node.names:
                imported_name = alias.asname or alias.name.split(".")[0]
                imports.append(alias.name)
                if imported_name not in referenced_names:
                    issues.append(
                        f"Possible unused import: {alias.name}"
                    )

        elif isinstance(node, ast.ImportFrom):
            for alias in node.names:
                imported_name = alias.asname or alias.name
                imports.append(
                    f"{node.module}.{alias.name}"
                    if node.module else alias.name
                )
                if imported_name not in referenced_names:
                    issues.append(
                        f"Possible unused import: {imported_name}"
                    )

    return as_json({
        "status": "success",
        "file": file_path,
        "syntax": "PASS",
        "function_count": len(functions),
        "functions": functions,
        "class_count": len(classes),
        "classes": classes,
        "imports": imports,
        "issue_count": len(issues),
        "issues": issues,
        "line_count": len(source.splitlines())
    })


@mcp.tool()
def run_tests() -> str:
    """Run pytest only in the sample project, with timeout protection."""
    start = time.perf_counter()
    try:
        result = subprocess.run(
            [
                sys.executable, "-m", "pytest",
                "-q", "--disable-warnings"
            ],
            cwd=str(PROJECT_DIR),
            capture_output=True,
            text=True,
            timeout=15
        )
        elapsed_ms = round(
            (time.perf_counter() - start) * 1000, 3
        )
        output = (result.stdout or "").strip()
        match = re.search(r"(\\d+) passed", output)

        return as_json({
            "status": "success" if result.returncode == 0 else "failed",
            "error_category": (
                None if result.returncode == 0 else "tool_error"
            ),
            "return_code": result.returncode,
            "execution_time_ms": elapsed_ms,
            "tests_passed": int(match.group(1)) if match else 0,
            "output": output,
            "error": (result.stderr or "").strip()
        })

    except subprocess.TimeoutExpired:
        return as_json({
            "status": "timeout",
            "error_category": "timeout",
            "message": "pytest exceeded 15 seconds"
        })

    except Exception as exc:
        return as_json({
            "status": "error",
            "error_category": "execution_error",
            "message": str(exc)
        })


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVER_FILE.write_text(server_code, encoding="utf-8")
print("\nMCP server file created:", SERVER_FILE)

# -------------------------
# 4. Planner, supervisor and policy layer
# -------------------------

USER_REQUEST = (
    "Check the pricing package for release readiness: inspect pricing.py, "
    "find TODOs, list the project files, read the README, and run tests "
    "after static analysis passes."
)

TOOL_ARGUMENTS = {
    "list_files": {},
    "read_file": {"file_path": str},
    "search_code": {"keyword": str},
    "analyze_python": {"file_path": str},
    "run_tests": {}
}

TOOL_POLICIES = {
    "list_files": {
        "risk": "LOW", "allowed": True
    },
    "read_file": {
        "risk": "LOW", "allowed": True
    },
    "search_code": {
        "risk": "LOW", "allowed": True
    },
    "analyze_python": {
        "risk": "LOW", "allowed": True
    },
    "run_tests": {
        "risk": "MEDIUM",
        "allowed": True,
        "scope": "Sample project only; 15-second timeout"
    }
}

RUN_TESTS_APPROVED = True


def planner_agent(user_request: str) -> list:
    """Convert the request into a deterministic MCP tool plan."""
    request = user_request.casefold()
    plan = []

    if "pricing.py" in request and (
        "inspect" in request or "analy" in request
    ):
        plan.append({
            "step": 1,
            "task_id": "TASK-001",
            "tool": "analyze_python",
            "arguments": {"file_path": "pricing.py"},
            "phase": "parallel",
            "depends_on": []
        })

    if "todo" in request:
        plan.append({
            "step": len(plan) + 1,
            "task_id": "TASK-002",
            "tool": "search_code",
            "arguments": {"keyword": "TODO"},
            "phase": "parallel",
            "depends_on": []
        })

    if "list" in request and "file" in request:
        plan.append({
            "step": len(plan) + 1,
            "task_id": "TASK-003",
            "tool": "list_files",
            "arguments": {},
            "phase": "parallel",
            "depends_on": []
        })

    if "readme" in request:
        plan.append({
            "step": len(plan) + 1,
            "task_id": "TASK-004",
            "tool": "read_file",
            "arguments": {"file_path": "README.md"},
            "phase": "parallel",
            "depends_on": []
        })

    if "run tests" in request or "run test" in request:
        plan.append({
            "step": len(plan) + 1,
            "task_id": "TASK-005",
            "tool": "run_tests",
            "arguments": {},
            "phase": "dependent",
            "depends_on": ["TASK-001"]
        })

    return plan


def supervisor_agent(plan: list, available_tools: list) -> dict:
    """Check tool discovery, arguments and path safety before execution."""
    approved, rejected = [], []
    available = set(available_tools)

    for task in plan:
        name = task.get("tool")
        args = task.get("arguments", {})
        reason = None

        if name not in available or name not in TOOL_ARGUMENTS:
            reason = "Tool is unavailable or not in the allowed schema."

        elif not isinstance(args, dict):
            reason = "Arguments must be a dictionary."

        elif set(args) != set(TOOL_ARGUMENTS[name]):
            reason = (
                f"Invalid arguments; expected keys: "
                f"{sorted(TOOL_ARGUMENTS[name])}"
            )

        else:
            for key, expected_type in TOOL_ARGUMENTS[name].items():
                if not isinstance(args.get(key), expected_type):
                    reason = f"Argument '{key}' must be {expected_type.__name__}."
                    break

            if reason is None and name in {"read_file", "analyze_python"}:
                relative = Path(args["file_path"])
                if relative.is_absolute() or ".." in relative.parts:
                    reason = "Unsafe project path rejected by supervisor."

        if reason:
            rejected.append({
                "task_id": task.get("task_id"),
                "tool": name,
                "reason": reason
            })
        else:
            approved.append(task)

    return {"approved": approved, "rejected": rejected}


def policy_check(task: dict) -> dict:
    """Apply risk-based tool policy."""
    policy = TOOL_POLICIES.get(task.get("tool"))

    if not policy or not policy.get("allowed", False):
        return {
            "allowed": False,
            "reason": "No permissive tool policy exists."
        }

    if task.get("tool") == "run_tests" and not RUN_TESTS_APPROVED:
        return {
            "allowed": False,
            "risk": "MEDIUM",
            "reason": "Test execution approval is disabled."
        }

    return {
        "allowed": True,
        "risk": policy["risk"],
        "scope": policy.get(
            "scope", "Project-scoped read-only operation"
        )
    }


def validator_agent(execution_results: list) -> dict:
    """Check whether required tool outputs satisfy workflow conditions."""
    by_tool = {item["tool"]: item for item in execution_results}

    analysis = by_tool.get("analyze_python", {}).get("output", {})
    search = by_tool.get("search_code", {}).get("output", {})
    files = by_tool.get("list_files", {}).get("output", {})
    readme = by_tool.get("read_file", {}).get("output", {})
    tests = by_tool.get("run_tests", {}).get("output", {})

    checks = {
        "static_analysis_succeeded": analysis.get("status") == "success",
        "python_syntax_valid": analysis.get("syntax") == "PASS",
        "project_files_found": all(
            name in files.get("files", [])
            for name in [
                "pricing.py", "app.py", "test_pricing.py", "README.md"
            ]
        ),
        "todo_search_found_expected_finding": (
            search.get("match_count", 0) >= 1
        ),
        "readme_read_succeeded": (
            readme.get("status") == "success"
            and "Release Readiness" in readme.get("content", "")
        ),
        "automated_tests_passed": (
            tests.get("status") == "success"
            and tests.get("return_code") == 0
            and tests.get("tests_passed", 0) > 0
        )
    }

    return {
        "checks": checks,
        "passed_checks": sum(bool(v) for v in checks.values()),
        "total_checks": len(checks),
        "overall_valid": all(checks.values())
    }

# -------------------------
# 5. MCP parsing, tracing and recovery
# -------------------------

TRACE_RECORDS = []
TRACE_ID = "TRACE-" + uuid.uuid4().hex[:8].upper()
MAX_ATTEMPTS = 2


def parse_call_result(response):
    raw_text = "\n".join(
        block.text for block in getattr(response, "content", [])
        if hasattr(block, "text")
    )

    if raw_text:
        try:
            payload = json.loads(raw_text)
        except json.JSONDecodeError:
            payload = {"raw_output": raw_text}
    else:
        structured = getattr(response, "structuredContent", None)
        payload = (
            structured if structured is not None
            else {"raw_output": "<empty tool response>"}
        )

    is_error = getattr(response, "isError", None)
    if is_error is None:
        is_error = getattr(response, "is_error", False)

    return payload, bool(is_error)


async def execute_mcp_tool(
    session, tool_name: str, arguments: dict,
    task_id: str, trace_records: list
):
    """Call an MCP tool, retry transient errors and record a trace."""
    started = time.perf_counter()
    attempts = 0
    payload = {}
    mcp_error = False
    exception_text = None

    while attempts < MAX_ATTEMPTS:
        attempts += 1

        try:
            response = await session.call_tool(
                tool_name, arguments=arguments
            )
            payload, mcp_error = parse_call_result(response)

            # Retry a tool-reported timeout only once.
            if (
                isinstance(payload, dict)
                and payload.get("status") == "timeout"
                and attempts < MAX_ATTEMPTS
            ):
                await asyncio.sleep(0.15 * attempts)
                continue

            exception_text = None
            break

        except Exception as exc:
            exception_text = f"{type(exc).__name__}: {exc}"
            exception_name = type(exc).__name__.casefold()

            retryable = (
                isinstance(
                    exc,
                    (TimeoutError, asyncio.TimeoutError,
                     ConnectionError, OSError)
                )
                or any(
                    token in exception_name
                    for token in (
                        "timeout", "closedresource", "brokenresource"
                    )
                )
            )

            if retryable and attempts < MAX_ATTEMPTS:
                logger.warning(
                    "Retryable MCP invocation error for %s; "
                    "attempt %s/%s: %s",
                    tool_name, attempts, MAX_ATTEMPTS, exception_text
                )
                await asyncio.sleep(0.15 * attempts)
                continue

            payload = {
                "status": "error",
                "error_category": "execution_error",
                "message": exception_text
            }
            break

    elapsed_ms = round(
        (time.perf_counter() - started) * 1000, 3
    )

    tool_status = (
        payload.get("status", "error")
        if isinstance(payload, dict) else "error"
    )

    if mcp_error:
        trace_status = "mcp_error"
    elif tool_status == "success":
        trace_status = "success"
    elif tool_status == "blocked":
        trace_status = "blocked"
    elif tool_status == "timeout":
        trace_status = "timeout"
    elif exception_text:
        trace_status = "error"
    else:
        trace_status = (
            "error" if tool_status in {"error", "failed"} else "success"
        )

    record = {
        "trace_id": TRACE_ID,
        "task_id": task_id,
        "tool": tool_name,
        "arguments": arguments,
        "status": trace_status,
        "tool_status": tool_status,
        "execution_time_ms": elapsed_ms,
        "attempts": attempts,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "error": (
            exception_text
            or (
                payload.get("message")
                if isinstance(payload, dict)
                and trace_status in {
                    "error", "timeout", "blocked", "mcp_error"
                }
                else None
            )
        ),
        "output": payload
    }

    trace_records.append(record)

    with TRACE_FILE.open("a", encoding="utf-8") as trace_file:
        trace_file.write(
            json.dumps(record, ensure_ascii=False, default=str) + "\n"
        )

    logger.info(
        "trace_id=%s task_id=%s tool=%s status=%s "
        "time_ms=%.3f attempts=%s",
        TRACE_ID, task_id, tool_name, trace_status,
        elapsed_ms, attempts
    )

    return record

# -------------------------
# 6. End-to-end MCP workflow
# -------------------------

async def run_experiment():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_FILE)]
    )

    with SERVER_STDERR_FILE.open("a", encoding="utf-8") as server_err:
        async with stdio_client(
            server_params, errlog=server_err
        ) as (read_stream, write_stream):

            async with ClientSession(
                read_stream, write_stream
            ) as session:

                await session.initialize()
                tool_response = await session.list_tools()
                available_tools = [
                    tool.name for tool in tool_response.tools
                ]

                print("\n[1] MCP CLIENT-SERVER INITIALIZATION")
                print("Transport: stdio | Session initialized: YES")
                print("Dynamically discovered tools:")

                for index, tool in enumerate(
                    tool_response.tools, start=1
                ):
                    print(
                        f"{index}. {tool.name} — "
                        f"{tool.description or 'No description supplied'}"
                    )

                print("\n[2] PLANNER AGENT")
                plan = planner_agent(USER_REQUEST)
                print("User request:", USER_REQUEST)
                print(json.dumps(plan, indent=2))

                print("\n[3] SUPERVISOR AND SAFETY POLICY")
                supervision = supervisor_agent(plan, available_tools)

                print("Approved:", len(supervision["approved"]))
                print("Rejected:", len(supervision["rejected"]))

                policy_decisions = {
                    task["task_id"]: policy_check(task)
                    for task in supervision["approved"]
                }

                for task_id, decision in policy_decisions.items():
                    print(
                        task_id,
                        "|", decision.get("risk", "UNKNOWN"),
                        "|", "ALLOW" if decision["allowed"] else "BLOCK",
                        "|", decision.get(
                            "scope", decision.get("reason", "")
                        )
                    )

                # Four independent tools execute concurrently over MCP.
                parallel_tasks = [
                    task for task in supervision["approved"]
                    if task.get("phase") == "parallel"
                    and policy_decisions[task["task_id"]]["allowed"]
                ]

                parallel_results = await asyncio.gather(*[
                    execute_mcp_tool(
                        session, task["tool"], task["arguments"],
                        task["task_id"], TRACE_RECORDS
                    )
                    for task in parallel_tasks
                ])

                by_task_id = {
                    task["task_id"]: result
                    for task, result in zip(
                        parallel_tasks, parallel_results
                    )
                }

                # Dependency gate: execute pytest only after valid AST analysis.
                analysis_record = by_task_id.get("TASK-001", {})
                analysis_output = analysis_record.get("output", {})

                analysis_gate = (
                    analysis_record.get("status") == "success"
                    and analysis_output.get("status") == "success"
                    and analysis_output.get("syntax") == "PASS"
                )

                print("\n[4] DEPENDENCY-AWARE EXECUTION")
                print(
                    "Phase 1: independent tools executed concurrently."
                )
                print("Degree of parallelism:", len(parallel_tasks))
                print(
                    "Dependency gate for run_tests:",
                    "PASS" if analysis_gate
                    else "FAIL — run_tests will be skipped"
                )

                for task in supervision["approved"]:
                    if task.get("phase") != "dependent":
                        continue

                    if not policy_decisions[task["task_id"]]["allowed"]:
                        by_task_id[task["task_id"]] = {
                            "task_id": task["task_id"],
                            "tool": task["tool"],
                            "status": "blocked",
                            "execution_time_ms": 0.0,
                            "attempts": 0,
                            "output": {
                                "status": "blocked",
                                "message": "Blocked by policy"
                            }
                        }

                    elif (
                        all(
                            dependency in {
                                result.get("task_id")
                                for result in by_task_id.values()
                            }
                            for dependency in task.get("depends_on", [])
                        )
                        and analysis_gate
                    ):
                        by_task_id[task["task_id"]] = (
                            await execute_mcp_tool(
                                session, task["tool"], task["arguments"],
                                task["task_id"], TRACE_RECORDS
                            )
                        )

                    else:
                        by_task_id[task["task_id"]] = {
                            "task_id": task["task_id"],
                            "tool": task["tool"],
                            "status": "skipped",
                            "execution_time_ms": 0.0,
                            "attempts": 0,
                            "output": {
                                "status": "skipped",
                                "message": "Dependency validation failed"
                            }
                        }

                main_results = [
                    by_task_id[task["task_id"]]
                    for task in supervision["approved"]
                    if task["task_id"] in by_task_id
                ]

                print("\n[5] MCP TOOL RESULTS")
                for result in main_results:
                    print(
                        f"\n{result['task_id']} | {result['tool']} | "
                        f"status={result['status']} | "
                        f"{result.get('execution_time_ms', 0):.3f} ms | "
                        f"attempts={result.get('attempts', 0)}"
                    )
                    print(json.dumps(
                        result.get("output", {}),
                        indent=2, ensure_ascii=False, default=str
                    ))

                print("\n[6] VALIDATOR AGENT")
                validation = validator_agent(main_results)

                for name, passed in validation["checks"].items():
                    print(
                        f"{'PASS' if passed else 'FAIL'} | {name}"
                    )

                print(
                    f"Validation: {validation['passed_checks']}/"
                    f"{validation['total_checks']} checks passed | "
                    f"overall_valid={validation['overall_valid']}"
                )

                # Negative tests: security, input error and unknown tool.
                print("\n[7] FAILURE HANDLING AND RECOVERY")

                security_probe = await execute_mcp_tool(
                    session, "read_file",
                    {"file_path": "../../etc/passwd"},
                    "SAFETY-001", TRACE_RECORDS
                )

                missing_probe = await execute_mcp_tool(
                    session, "read_file",
                    {"file_path": "missing_file.py"},
                    "SAFETY-002", TRACE_RECORDS
                )

                unknown_tool_plan = [{
                    "step": 99,
                    "task_id": "SAFETY-003",
                    "tool": "delete_repository",
                    "arguments": {}
                }]

                unknown_tool_result = supervisor_agent(
                    unknown_tool_plan, available_tools
                )

                safety_checks = {
                    "path_traversal_blocked": (
                        security_probe.get("output", {}).get("status")
                        == "blocked"
                        and security_probe.get("output", {}).get(
                            "error_category"
                        ) == "security_error"
                    ),
                    "missing_file_stops_without_retry": (
                        missing_probe.get("output", {}).get(
                            "error_category"
                        ) == "input_error"
                        and missing_probe.get("attempts") == 1
                    ),
                    "unknown_tool_rejected_before_execution": (
                        len(unknown_tool_result["rejected"]) == 1
                        and not unknown_tool_result["approved"]
                    )
                }

                print(
                    "Path traversal:",
                    json.dumps(security_probe.get("output", {}))
                )
                print(
                    "Missing file:",
                    json.dumps(missing_probe.get("output", {})),
                    "| attempts:", missing_probe.get("attempts")
                )
                print(
                    "Unknown tool:",
                    json.dumps(unknown_tool_result)
                )

                for name, passed in safety_checks.items():
                    print(
                        f"{'PASS' if passed else 'FAIL'} | {name}"
                    )

                # -------------------------
                # 7. Sequential vs parallel benchmark
                # -------------------------
                print("\n[8] SEQUENTIAL VS PARALLEL PERFORMANCE")

                benchmark_tasks = [
                    ("analyze_python", {"file_path": "pricing.py"}),
                    ("search_code", {"keyword": "TODO"}),
                    ("list_files", {}),
                    ("read_file", {"file_path": "README.md"})
                ]

                sequential_ms, parallel_ms = [], []
                repetitions = 5

                for run_no in range(1, repetitions + 1):
                    seq_start = time.perf_counter()

                    for idx, (name, args) in enumerate(
                        benchmark_tasks, start=1
                    ):
                        await execute_mcp_tool(
                            session, name, args,
                            f"PERF-S{run_no}-{idx}", TRACE_RECORDS
                        )

                    sequential_ms.append(
                        (time.perf_counter() - seq_start) * 1000
                    )

                    par_start = time.perf_counter()

                    await asyncio.gather(*[
                        execute_mcp_tool(
                            session, name, args,
                            f"PERF-P{run_no}-{idx}", TRACE_RECORDS
                        )
                        for idx, (name, args) in enumerate(
                            benchmark_tasks, start=1
                        )
                    ])

                    parallel_ms.append(
                        (time.perf_counter() - par_start) * 1000
                    )

                average_seq = statistics.mean(sequential_ms)
                average_par = statistics.mean(parallel_ms)

                reduction = (
                    (average_seq - average_par) / average_seq * 100
                    if average_seq else 0.0
                )

                performance = {
                    "independent_tool_calls_per_run": len(benchmark_tasks),
                    "repetitions": repetitions,
                    "sequential_times_ms": [
                        round(x, 3) for x in sequential_ms
                    ],
                    "parallel_times_ms": [
                        round(x, 3) for x in parallel_ms
                    ],
                    "average_sequential_ms": round(average_seq, 3),
                    "average_parallel_ms": round(average_par, 3),
                    "average_time_reduction_percent": round(reduction, 2),
                    "parallel_faster_on_average": average_par < average_seq,
                    "degree_of_parallelism": len(benchmark_tasks)
                }

                print(
                    "Run | Sequential (ms) | Parallel (ms) | Difference (ms)"
                )
                print(
                    "----|-----------------|---------------|----------------"
                )

                for idx, (seq, par) in enumerate(
                    zip(sequential_ms, parallel_ms), start=1
                ):
                    print(
                        f"{idx:>3} | {seq:>15.3f} | "
                        f"{par:>13.3f} | {seq-par:>14.3f}"
                    )

                print(f"Average sequential: {average_seq:.3f} ms")
                print(f"Average parallel:   {average_par:.3f} ms")
                print(
                    f"Average time reduction: {reduction:.2f}% "
                    "(negative means parallel was slower)"
                )

                # -------------------------
                # 8. Generate reports
                # -------------------------
                final_status = (
                    "VALIDATED"
                    if (
                        validation["overall_valid"]
                        and all(safety_checks.values())
                        and not supervision["rejected"]
                    )
                    else "REQUIRES_REVIEW"
                )

                workflow_summary = {
                    "planned_tasks": len(plan),
                    "approved_tasks": len(supervision["approved"]),
                    "rejected_tasks": len(supervision["rejected"]),
                    "main_tasks_executed": len(main_results),
                    "main_successful_tasks": sum(
                        1 for item in main_results
                        if item.get("status") == "success"
                    ),
                    "main_failed_or_skipped_tasks": sum(
                        1 for item in main_results
                        if item.get("status") != "success"
                    ),
                    "parallel_phase_task_ids": [
                        task["task_id"] for task in parallel_tasks
                    ],
                    "dependency_gate_passed": analysis_gate
                }

                report = {
                    "experiment": "Experiment 06",
                    "title": (
                        "Agentic Developer Release-Readiness System using MCP"
                    ),
                    "domain": "Developer Tools",
                    "custom_use_case": (
                        "Pre-release quality checks for a Python pricing package"
                    ),
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "user_request": USER_REQUEST,
                    "mcp": {
                        "transport": "stdio",
                        "session_initialized": True,
                        "discovered_tools": available_tools
                    },
                    "planning": {"plan": plan},
                    "supervision": supervision,
                    "policy_decisions": policy_decisions,
                    "execution": {
                        "summary": workflow_summary,
                        "tasks": main_results
                    },
                    "validation": validation,
                    "failure_handling": {
                        "safety_checks": safety_checks,
                        "path_traversal_probe": security_probe,
                        "missing_file_probe": missing_probe,
                        "unknown_tool_probe": unknown_tool_result,
                        "retry_policy": (
                            "Retry transient invocation errors and tool-reported "
                            "timeouts once; do not retry input or security errors."
                        )
                    },
                    "performance": performance,
                    "observability": {
                        "trace_id": TRACE_ID,
                        "trace_records_in_this_run": len(TRACE_RECORDS),
                        "trace_file": str(TRACE_FILE),
                        "execution_log": str(LOG_FILE)
                    },
                    "final_status": final_status
                }

                summary = {
                    "experiment": report["experiment"],
                    "title": report["title"],
                    "final_status": final_status,
                    "trace_id": TRACE_ID,
                    "mcp_tools_discovered": available_tools,
                    "workflow": workflow_summary,
                    "validation_passed": validation["overall_valid"],
                    "safety_checks_passed": all(safety_checks.values()),
                    "performance": performance,
                    "artifacts": {
                        "execution_log": str(LOG_FILE),
                        "mcp_traces_jsonl": str(TRACE_FILE),
                        "developer_automation_report": str(REPORT_FILE),
                        "complete_report": str(COMPLETE_REPORT_FILE),
                        "execution_summary": str(SUMMARY_FILE),
                        "artifact_zip": str(ARTIFACT_ZIP)
                    }
                }

                for path, data in [
                    (REPORT_FILE, report),
                    (COMPLETE_REPORT_FILE, report),
                    (SUMMARY_FILE, summary)
                ]:
                    path.write_text(
                        json.dumps(
                            data, indent=2, ensure_ascii=False, default=str
                        ),
                        encoding="utf-8"
                    )

                # Bundle all artifacts into a ZIP archive.
                if ARTIFACT_ZIP.exists():
                    ARTIFACT_ZIP.unlink()

                with zipfile.ZipFile(
                    ARTIFACT_ZIP, "w",
                    compression=zipfile.ZIP_DEFLATED
                ) as bundle:
                    for path in sorted(PROJECT_DIR.iterdir()):
                        if path.is_file():
                            bundle.write(
                                path,
                                arcname=f"sample_project/{path.name}"
                            )

                    for path in [
                        LOG_FILE, TRACE_FILE, SERVER_STDERR_FILE,
                        REPORT_FILE, COMPLETE_REPORT_FILE, SUMMARY_FILE,
                        SERVER_FILE
                    ]:
                        if path.exists():
                            bundle.write(
                                path,
                                arcname=f"experiment_06_artifacts/{path.name}"
                            )

                print("\n[9] FINAL END-TO-END SUMMARY")
                print(json.dumps(
                    summary, indent=2, ensure_ascii=False, default=str
                ))

                print("\n[10] GENERATED ARTIFACTS")
                for path in [
                    LOG_FILE, TRACE_FILE, REPORT_FILE,
                    COMPLETE_REPORT_FILE, SUMMARY_FILE, ARTIFACT_ZIP
                ]:
                    print(
                        f"{'OK' if path.exists() else 'MISSING'} | {path} "
                        f"| {path.stat().st_size if path.exists() else 0} bytes"
                    )

                print(
                    "\nUse the Colab Files pane to access the ZIP and JSON reports."
                )
                return report


FINAL_REPORT = await run_experiment()

# -------------------------
# 9. Verify generated reports
# -------------------------

with COMPLETE_REPORT_FILE.open("r", encoding="utf-8") as report_file:
    VERIFIED_REPORT = json.load(report_file)

with SUMMARY_FILE.open("r", encoding="utf-8") as summary_file:
    VERIFIED_SUMMARY = json.load(summary_file)

assert VERIFIED_REPORT["final_status"] in {
    "VALIDATED", "REQUIRES_REVIEW"
}
assert VERIFIED_SUMMARY["trace_id"] == TRACE_ID
assert TRACE_FILE.exists()
assert REPORT_FILE.exists()
assert ARTIFACT_ZIP.exists()

print("\nREPORT VERIFICATION: SUCCESS")
print("Final status:", VERIFIED_REPORT["final_status"])
print("Trace records:", len(TRACE_RECORDS))
print("Artifact ZIP:", ARTIFACT_ZIP)

EXPERIMENT 06: AGENTIC DEVELOPER RELEASE-READINESS SYSTEM
Domain: Developer Tools
Use case: Python pricing package release-readiness checks
Project: /content/release_readiness_project
Files: ['README.md', 'app.py', 'pricing.py', 'test_pricing.py']

MCP server file created: /content/mcp_release_readiness_server.py

[1] MCP CLIENT-SERVER INITIALIZATION
Transport: stdio | Session initialized: YES
Dynamically discovered tools:
1. list_files — List files in the sample release-readiness project.
2. read_file — Read a text file inside the sample project.
3. search_code — Search for a keyword across Python project files.
4. analyze_python — Run AST-based syntax and code-quality analysis.
5. run_tests — Run pytest only in the sample project, with timeout protection.

[2] PLANNER AGENT
User request: Check the pricing package for release readiness: inspect pricing.py, find TODOs, list the project files, read the README, and run tests after static analysis passes.
[
  {
    "step": 1,
    "task_id